In [ ]:
"""This module implements an XRD pattern calculator."""

from __future__ import annotations
import torch
import json
import os
from math import asin, cos, degrees, pi, radians, sin
from typing import TYPE_CHECKING

import numpy as np

# Preliminary steps
# XRD wavelengths in angstroms
with open("atomic_scattering_params.json") as f:
	ATOMIC_SCATTERING_PARAMS = json.load(f)

import pandas as pd

periodic_csv = pd.read_csv("Periodic Table of Elements.csv")

len(list(ATOMIC_SCATTERING_PARAMS.keys()))

atom_names = {(index+1):symbol for index,symbol in enumerate(list(periodic_csv['Symbol']))}

atom_names

In [ ]:
def get_points_in_sphere(recip_latt, max_r):
	recip_pts = []
	max_h = round(max_r / float(torch.linalg.norm(recip_latt[0])))
	max_k = round(max_r / float(torch.linalg.norm(recip_latt[1])))
	max_l = round(max_r / float(torch.linalg.norm(recip_latt[2])))
	for h in range(max_h):
		for k in range(max_k):
			r = torch.linalg.norm(h * recip_latt[0] + k * recip_latt[1])
			if r > max_r:
				break
			for l in range(max_l):
				r = torch.linalg.norm(h * recip_latt[0] + k * recip_latt[1] + l * recip_latt[2])
				if r > max_r:
					break
				recip_pts.append([[h,k,l],r])

	return recip_pts

def dif_diffraction_calc(lengths, angles, atoms, wavelength = 1.54184, scaled = True, two_theta_range=(0, 90), hex_angle_tol = 0.01, debye_waller_factors=None):
    angles = torch.tensor(angles, requires_grad=True)
    lengths = torch.tensor(lengths, requires_grad=True)
    angles_rad = angles * np.pi / 180
    a_vector = torch.tensor([1, 0, 0]) * lengths[0]

    b_vector = (torch.tensor([1,0,0]) * torch.cos(angles_rad[2]) + torch.tensor([0,1,0]) * torch.sin(angles_rad[2])) * lengths[1]

    c_vector = (torch.tensor([1,0,0]) * torch.cos(angles_rad[1]) + torch.tensor([0,1,0]) * ((torch.cos(angles_rad[0]) - torch.cos(angles_rad[1]) * torch.cos(angles_rad[2])) /  torch.sin(angles_rad[2])) + torch.tensor([0,0,1]) * torch.sqrt(1 - torch.square(torch.cos(angles_rad[1])) - torch.square((torch.cos(angles_rad[0]) - torch.cos(angles_rad[1]) * torch.cos(angles_rad[2])) / torch.sin(angles_rad[2]))) )* lengths[2]

    #ask Tsach, does it make sense to access an element of the list of angles_rad or just apply the operation on the whole tensor
    #should I call all the torch.cos or just do cos


    # Collect the debye_waller_factors
    debye_waller_factors = debye_waller_factors or {}

    # Obtained from Bragg condition. Note that reciprocal lattice
    # vector length is 1 / d_hkl.
    min_r, max_r = (
        (0, 2 / wavelength)
        if two_theta_range is None
        else [2 * sin(radians(t / 2)) / wavelength for t in two_theta_range]
    )

    cell_matrix = torch.stack((a_vector,b_vector, c_vector))

    # Obtain crystallographic reciprocal lattice points within range
    recip_latt = torch.linalg.inv(cell_matrix).T

    recip_pts = get_points_in_sphere(recip_latt, max_r)

    if min_r:
        recip_pts = [pt for pt in recip_pts if pt[1] >= min_r]

    # Collect all the information relevant to each atom
    zs = torch.tensor([i[0] for i in atoms])
    fcoords = torch.tensor([i[1:] for i in atoms])
    try:
        coeffs = torch.tensor([ATOMIC_SCATTERING_PARAMS[atom_names.get(int(zs[0]))]])
    except KeyError:
        raise ValueError(
            f"Unable to calculate XRD pattern as there is no scattering coefficients for {int(z)}."
        )

    #Check if any debye_waller_factors are included
    if debye_waller_factors == {}:
        dwfactors_cor = False
    else:
        dwfactors_cor = True

    dwfactors_cor = True

    if dwfactors_cor:
        dwfactors = torch.tensor(debye_waller_factors.get(atom_names.get(int(zs[0])), [0]))

    for z in zs[1:]:
        try:
            c = torch.tensor([ATOMIC_SCATTERING_PARAMS[atom_names.get(int(z))]])
        except KeyError:
            raise ValueError(
                f"Unable to calculate XRD pattern as there is no scattering coefficients for {int(z)}."
            )
        coeffs = torch.cat((coeffs,c))
        if dwfactors_cor:
            dwfactors = torch.cat((dwfactors, debye_waller_factors.get(atom_names.get(int(z)), torch.tensor([0]))))


    peaks_ti = torch.tensor([[0,0]])
    two_thetas = torch.tensor(0).reshape(1)


    for hkl, g_hkl in sorted(recip_pts, key=lambda i: (i[1], -i[0][0], -i[0][1], -i[0][2])):
        # Force miller indices to be integers.
        hkl = torch.tensor([hkl], dtype = torch.float)
        if g_hkl != 0:
            # Bragg condition
            theta = torch.asin(wavelength * g_hkl / 2)



            # s = sin(theta) / wavelength = 1 / 2d = |ghkl| / 2 (d =
            # 1/|ghkl|)
            s = g_hkl / 2

            # Store s^2 since we are using it a few times.
            s2 = s**2

            # Vectorized computation of g.r for all fractional coords and
            # hkl.
            g_dot_r = torch.tensordot(fcoords, torch.transpose(hkl,0,1), dims = 1).T[0]


            # Highly vectorized computation of atomic scattering factors.
            # Equivalent non-vectorized code is::
            #
            #   for site in structure:
            #	  el = site.specie
            #	  coeff = ATOMIC_SCATTERING_PARAMS[el.symbol]
            #	  fs = el.Z - 41.78214 * s2 * sum(
            #		  [d[0] * exp(-d[1] * s2) for d in coeff])
            fs = zs - 41.78214 * s2 * torch.sum(
                coeffs[:, :, 0] * torch.exp(-coeffs[:, :, 1] * s2),
                axis=1,  # type: ignore
            )

            dw_correction = torch.exp(-dwfactors * s2)

            # Structure factor = sum of atomic scattering factors (with
            # position factor exp(2j * pi * g.r and occupancies).
            # Vectorized computation.
            f_hkl = torch.sum(fs * torch.exp(2j * pi * g_dot_r) * dw_correction)

            # Lorentz polarization correction for hkl
            lorentz_factor = (1 + torch.cos(2 * theta) ** 2) / (torch.sin(theta) ** 2 * torch.cos(theta))

            # Intensity for hkl is modulus square of structure factor.
            i_hkl = (f_hkl * torch.conj(f_hkl)).real

            two_theta = (2 * theta * 180 / np.pi).reshape(1)

            # Deal with floating point precision issues. This combines all peaks within two_theta_tol
            two_theta_tol = 1e-5
            ind = torch.where(
                torch.abs(torch.subtract(two_thetas, two_theta)) < two_theta_tol
            )

            if len(ind[0]) > 0:
                #peaks[ind] += i_hkl * lorentz_factor
                #peaks[two_thetas[ind[0][0]]][1].append(tuple(hkl))  # type: ignore
                i = 0
            else:
                d_hkl = 1 / g_hkl
                peak_intensity = (i_hkl * lorentz_factor).reshape(1)
                peak = torch.cat((two_theta,peak_intensity)).reshape(1,2)
                print(peak)
                peaks_ti = torch.cat((peaks_ti,peak))
                two_thetas = torch.cat((two_thetas,two_theta))

    print("the peaks are")
    print(peaks_ti)
    peaks_ti[2][1].backward()
    print(hkl)
    print(g_hkl)
    print(zs.grad)
    print(angles.grad)
    print(lengths.grad)
    
    return(peaks_ti)


angles = [50.0,50.0,50.0]
lengths = [4.0,4.0,4.0]
atoms = [[1,0,0,0],[1,.5,.5,.5]]
result = dif_diffraction_calc(lengths, angles, atoms, wavelength = 1.54184, scaled = True, two_theta_range=(0, 90), hex_angle_tol = 0.001, debye_waller_factors=None)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
result.detach().numpy()[:,0]

In [ ]:
plt.stem(result.detach().numpy()[:,0],result.detach().numpy()[:,1])
plt.xlim([30,75])

In [ ]:
"""This module implements an XRD pattern calculator."""

from __future__ import annotations

import json
import os
from math import asin, cos, degrees, pi, radians, sin
from typing import TYPE_CHECKING

import numpy as np

from pymatgen.analysis.diffraction.core import (
    AbstractDiffractionPatternCalculator,
    DiffractionPattern,
    get_unique_families,
)
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer

if TYPE_CHECKING:
    from pymatgen.core import Structure

# XRD wavelengths in angstroms
WAVELENGTHS = {
    "CuKa": 1.54184,
    "CuKa2": 1.54439,
    "CuKa1": 1.54056,
    "CuKb1": 1.39222,
    "MoKa": 0.71073,
    "MoKa2": 0.71359,
    "MoKa1": 0.70930,
    "MoKb1": 0.63229,
    "CrKa": 2.29100,
    "CrKa2": 2.29361,
    "CrKa1": 2.28970,
    "CrKb1": 2.08487,
    "FeKa": 1.93735,
    "FeKa2": 1.93998,
    "FeKa1": 1.93604,
    "FeKb1": 1.75661,
    "CoKa": 1.79026,
    "CoKa2": 1.79285,
    "CoKa1": 1.78896,
    "CoKb1": 1.63079,
    "AgKa": 0.560885,
    "AgKa2": 0.563813,
    "AgKa1": 0.559421,
    "AgKb1": 0.497082,
}


class XRDCalculator(AbstractDiffractionPatternCalculator):
    r"""
    Computes the XRD pattern of a crystal structure.

    This code is implemented by Shyue Ping Ong as part of UCSD's NANO106 -
    Crystallography of Materials. The formalism for this code is based on
    that given in Chapters 11 and 12 of Structure of Materials by Marc De
    Graef and Michael E. McHenry. This takes into account the atomic
    scattering factors and the Lorentz polarization factor, but not
    the Debye-Waller (temperature) factor (for which data is typically not
    available). Note that the multiplicity correction is not needed since
    this code simply goes through all reciprocal points within the limiting
    sphere, which includes all symmetrically equivalent facets. The algorithm
    is as follows

    1. Calculate reciprocal lattice of structure. Find all reciprocal points
       within the limiting sphere given by \frac{2}{\lambda}.

    2. For each reciprocal point \mathbf{g_{hkl}} corresponding to
       lattice plane (hkl), compute the Bragg condition
       \sin(\theta) = \frac{ \lambda}{2d_{hkl}}

    3. Compute the structure factor as the sum of the atomic scattering
       factors. The atomic scattering factors are given by

           f(s) = Z - 41.78214 \times s^2 \times \sum \limits_{i=1}^n a_i \exp(-b_is^2)

       where s = \ frac{\ sin(\ theta)}{\ lambda} and a_i
       and b_i are the fitted parameters for each element. The
       structure factor is then given by

           F_{hkl} = \sum \limits_{j=1}^N f_j  \exp(2 \pi i  \mathbf{g_{hkl}} \cdot  \mathbf{r})

    4. The intensity is then given by the modulus square of the structure factor.

           I_{hkl} = F_{hkl}F_{hkl}^*

    5. Finally, the Lorentz polarization correction factor is applied. This
       factor is given by:

           P(\theta) = \frac{1 + \cos^2(2 \theta)}{\sin^2(\theta) \cos(\theta)}
    """

    # Tuple of available radiation keywords.
    AVAILABLE_RADIATION = tuple(WAVELENGTHS)

    def __init__(self, wavelength="CuKa", symprec: float = 0, debye_waller_factors=None):
        """
        Initializes the XRD calculator with a given radiation.

        Args:
            wavelength (str/float): The wavelength can be specified as either a
                float or a string. If it is a string, it must be one of the
                supported definitions in the AVAILABLE_RADIATION class
                variable, which provides useful commonly used wavelengths.
                If it is a float, it is interpreted as a wavelength in
                angstroms. Defaults to "CuKa", i.e, Cu K_alpha radiation.
            symprec (float): Symmetry precision for structure refinement. If
                set to 0, no refinement is done. Otherwise, refinement is
                performed using spglib with provided precision.
            debye_waller_factors ({element symbol: float}): Allows the
                specification of Debye-Waller factors. Note that these
                factors are temperature dependent.
        """
        if isinstance(wavelength, (float, int)):
            self.wavelength = wavelength
        elif isinstance(wavelength, str):
            self.radiation = wavelength
            self.wavelength = WAVELENGTHS[wavelength]
        else:
            raise TypeError(f"{type(wavelength)=} must be either float, int or str")
        self.symprec = symprec
        self.debye_waller_factors = debye_waller_factors or {}

    def get_pattern(self, structure: Structure, scaled=True, two_theta_range=(0, 90)):
        """
        Calculates the diffraction pattern for a structure.

        Args:
            structure (Structure): Input structure
            scaled (bool): Whether to return scaled intensities. The maximum
                peak is set to a value of 100. Defaults to True. Use False if
                you need the absolute values to combine XRD plots.
            two_theta_range ([float of length 2]): Tuple for range of
                two_thetas to calculate in degrees. Defaults to (0, 90). Set to
                None if you want all diffracted beams within the limiting
                sphere of radius 2 / wavelength.

        Returns:
            (XRDPattern)
        """
        # if self.symprec:
        #     finder = SpacegroupAnalyzer(structure, symprec=self.symprec)
        #     structure = finder.get_refined_structure()

        wavelength = self.wavelength
        latt = structure.lattice
        is_hex = latt.is_hexagonal()

        # Obtained from Bragg condition. Note that reciprocal lattice
        # vector length is 1 / d_hkl.
        min_r, max_r = (
            (0, 2 / wavelength)
            if two_theta_range is None
            else [2 * sin(radians(t / 2)) / wavelength for t in two_theta_range]
        )

        # Obtain crystallographic reciprocal lattice points within range
        recip_latt = latt.reciprocal_lattice_crystallographic
        recip_pts = recip_latt.get_points_in_sphere([[0, 0, 0]], [0, 0, 0], max_r)
        if min_r:
            recip_pts = [pt for pt in recip_pts if pt[1] >= min_r]

        # Create a flattened array of zs, coeffs, fcoords and occus. This is used to perform
        # vectorized computation of atomic scattering factors later. Note that these are not
        # necessarily the same size as the structure as each partially occupied specie occupies its
        # own position in the flattened array.
        _zs = []
        _coeffs = []
        _fcoords = []
        _occus = []
        _dwfactors = []

        for site in structure:
            for sp, occu in site.species.items():
                _zs.append(sp.Z)
                try:
                    c = ATOMIC_SCATTERING_PARAMS[sp.symbol]
                except KeyError:
                    raise ValueError(
                        f"Unable to calculate XRD pattern as there is no scattering coefficients for {sp.symbol}."
                    )
                _coeffs.append(c)
                _dwfactors.append(self.debye_waller_factors.get(sp.symbol, 0))
                _fcoords.append(site.frac_coords)
                _occus.append(occu)

        zs = np.array(_zs)
        coeffs = np.array(_coeffs)
        fcoords = np.array(_fcoords)
        occus = np.array(_occus)
        dwfactors = np.array(_dwfactors)
        peaks: dict[float, list[float | list[tuple[int, ...]]]] = {}
        two_thetas: list[float] = []

        for hkl, g_hkl, ind, _ in sorted(recip_pts, key=lambda i: (i[1], -i[0][0], -i[0][1], -i[0][2])):
            # Force miller indices to be integers.
            hkl = [int(round(i)) for i in hkl]
            if g_hkl != 0:
                # Bragg condition
                theta = asin(wavelength * g_hkl / 2)

                # s = sin(theta) / wavelength = 1 / 2d = |ghkl| / 2 (d =
                # 1/|ghkl|)
                s = g_hkl / 2

                # Store s^2 since we are using it a few times.
                s2 = s**2

                # Vectorized computation of g.r for all fractional coords and
                # hkl.
                g_dot_r = np.dot(fcoords, np.transpose([hkl])).T[0]

                # Highly vectorized computation of atomic scattering factors.
                # Equivalent non-vectorized code is::
                #
                #   for site in structure:
                #      el = site.specie
                #      coeff = ATOMIC_SCATTERING_PARAMS[el.symbol]
                #      fs = el.Z - 41.78214 * s2 * sum(
                #          [d[0] * exp(-d[1] * s2) for d in coeff])
                fs = zs - 41.78214 * s2 * np.sum(
                    coeffs[:, :, 0] * np.exp(-coeffs[:, :, 1] * s2),
                    axis=1,  # type: ignore
                )

                dw_correction = np.exp(-dwfactors * s2)

                # Structure factor = sum of atomic scattering factors (with
                # position factor exp(2j * pi * g.r and occupancies).
                # Vectorized computation.
                f_hkl = np.sum(fs * occus * np.exp(2j * pi * g_dot_r) * dw_correction)

                # Lorentz polarization correction for hkl
                lorentz_factor = (1 + cos(2 * theta) ** 2) / (sin(theta) ** 2 * cos(theta))

                # Intensity for hkl is modulus square of structure factor.
                i_hkl = (f_hkl * f_hkl.conjugate()).real

                two_theta = degrees(2 * theta)

                if is_hex:
                    # Use Miller-Bravais indices for hexagonal lattices.
                    hkl = (hkl[0], hkl[1], -hkl[0] - hkl[1], hkl[2])
                # Deal with floating point precision issues.
                ind = np.where(
                    np.abs(np.subtract(two_thetas, two_theta)) < AbstractDiffractionPatternCalculator.TWO_THETA_TOL
                )
                if len(ind[0]) > 0:
                    peaks[two_thetas[ind[0][0]]][0] += i_hkl * lorentz_factor
                    peaks[two_thetas[ind[0][0]]][1].append(tuple(hkl))  # type: ignore
                else:
                    d_hkl = 1 / g_hkl
                    peaks[two_theta] = [i_hkl * lorentz_factor, [tuple(hkl)], d_hkl]
                    two_thetas.append(two_theta)

        # Scale intensities so that the max intensity is 100.
        max_intensity = max(v[0] for v in peaks.values())
        x = []
        y = []
        hkls = []
        d_hkls = []
        for k in sorted(peaks):
            v = peaks[k]
            fam = get_unique_families(v[1])
            if v[0] / max_intensity * 100 > AbstractDiffractionPatternCalculator.SCALED_INTENSITY_TOL:  # type: ignore
                x.append(k)
                y.append(v[0])
                hkls.append([{"hkl": hkl, "multiplicity": mult} for hkl, mult in fam.items()])
                d_hkls.append(v[2])
        print('hkls')
        print(hkls)
        print('x')
        print(x)
        print('y')
        print(y)
        print('d_hkls')
        print(d_hkls)
        xrd = DiffractionPattern(x, y, hkls, d_hkls)
        if scaled:
            xrd.normalize(mode="max", value=100)
        return xrd


In [ ]:
from pymatgen.core import Lattice, Structure
import matplotlib.pyplot as plt

In [ ]:
result.detach().numpy()[:,0]

In [ ]:
result.detach().numpy()

In [ ]:
def get_points_in_sphere(recip_latt, max_r):
	recip_pts = []
	max_h = round(max_r / np.linalg.norm(recip_latt[0]))
	max_k = round(max_r / np.linalg.norm(recip_latt[1]))
	max_l = round(max_r / np.linalg.norm(recip_latt[2]))
	for h in range(max_h):
		for k in range(max_k):
			r = np.linalg.norm(h * recip_latt[0] + k * recip_latt[1])
			if r > max_r:
				break
			for l in range(max_l):
				r = np.linalg.norm(h * recip_latt[0] + k * recip_latt[1] + l * recip_latt[2])
				if r > max_r:
					break
				recip_pts.append([[h,k,l],r])

	return recip_pts


r"""
Computes the XRD pattern of a crystal structure.

This code is implemented by Shyue Ping Ong as part of UCSD's NANO106 -
Crystallography of Materials. The formalism for this code is based on
that given in Chapters 11 and 12 of Structure of Materials by Marc De
Graef and Michael E. McHenry. This takes into account the atomic
scattering factors and the Lorentz polarization factor, but not
the Debye-Waller (temperature) factor (for which data is typically not
available). Note that the multiplicity correction is not needed since
this code simply goes through all reciprocal points within the limiting
sphere, which includes all symmetrically equivalent facets. The algorithm
is as follows

1. Calculate reciprocal lattice of structure. Find all reciprocal points
   within the limiting sphere given by \frac{2}{\lambda}.

2. For each reciprocal point \mathbf{g_{hkl}} corresponding to
   lattice plane (hkl), compute the Bragg condition
   \sin(\theta) = \frac{ \lambda}{2d_{hkl}}

3. Compute the structure factor as the sum of the atomic scattering
   factors. The atomic scattering factors are given by

	   f(s) = Z - 41.78214 \times s^2 \times \sum \limits_{i=1}^n a_i \exp(-b_is^2)

   where s = \ frac{\ sin(\ theta)}{\ lambda} and a_i
   and b_i are the fitted parameters for each element. The
   structure factor is then given by

	   F_{hkl} = \sum \limits_{j=1}^N f_j  \exp(2 \pi i  \mathbf{g_{hkl}} \cdot  \mathbf{r})

4. The intensity is then given by the modulus square of the structure factor.

	   I_{hkl} = F_{hkl}F_{hkl}^*

5. Finally, the Lorentz polarization correction factor is applied. This
   factor is given by:

	   P(\theta) = \frac{1 + \cos^2(2 \theta)}{\sin^2(\theta) \cos(\theta)}
"""

# Tuple of available radiation keywords.
def get_pattern(lengths, angles, atoms, wavelength = 1.54184, scaled = True, two_theta_range=(0, 90), hex_angle_tol = 0.01, debye_waller_factors=None):
	"""
	Calculates the diffraction pattern for a structure.

	Args:
		structure (Structure): Input structure
		scaled (bool): Whether to return scaled intensities. The maximum
			peak is set to a value of 100. Defaults to True. Use False if
			you need the absolute values to combine XRD plots.
		two_theta_range ([float of length 2]): Tuple for range of
			two_thetas to calculate in degrees. Defaults to (0, 90). Set to
			None if you want all diffracted beams within the limiting
			sphere of radius 2 / wavelength.
		debye_waller_factors ({element symbol: float}): Allows the
			specification of Debye-Waller factors. Note that these
			factors are temperature dependent.
		Returns:
		(XRDPattern)
	"""

	#write the matrix for the cell
	angles = np.array(angles) * np.pi / 180
	a_vector = np.array([1, 0, 0]) * lengths[0]
	b_vector = np.array([cos(angles[2]), sin(angles[2]), 0]) * lengths[1]
	c_vector = np.array(
		[cos(angles[1]),
		(cos(angles[1]) - cos(angles[1]) * cos(angles[2]))/sin(angles[2]),
		np.sqrt(1 - cos(angles[1])**2 - ((cos(angles[1]) - cos(angles[1]) * cos(angles[2]))/sin(angles[2]))**2 )]
		) * lengths[2]

	cell_matrix = np.array([a_vector, b_vector, c_vector])

	#collect debye_waller factors
	debye_waller_factors = debye_waller_factors or {}

	#check if cell is hexagonal
	right_angles = [i for i in range(3) if abs(angles[i] - 90) < hex_angle_tol]
	hex_angles = [
	i for i in range(3) if abs(angles[i] - 60) < hex_angle_tol or abs(angles[i] - 120) < hex_angle_tol
	]
	is_hex = (len(right_angles) == 2
		and len(hex_angles) == 1
		and abs(lengths[right_angles[0]] - lengths[right_angles[1]]) < hex_length_tol)

	# Obtained from Bragg condition. Note that reciprocal lattice
	# vector length is 1 / d_hkl.
	min_r, max_r = (
		(0, 2 / wavelength)
		if two_theta_range is None
		else [2 * sin(radians(t / 2)) / wavelength for t in two_theta_range]
	)

	# Obtain crystallographic reciprocal lattice points within range
	recip_latt = np.linalg.inv(cell_matrix).T

	recip_pts = get_points_in_sphere(recip_latt, max_r)
#	recip_pts = [[[1,2,3],0.5], [[1,3,3],.3], [[1,1,3],0.5]]

	if min_r:
		recip_pts = [pt for pt in recip_pts if pt[1] >= min_r]

	# Create a flattened array of zs, coeffs, fcoords and occus. This is used to perform
	# vectorized computation of atomic scattering factors later. Note that these are not
	# necessarily the same size as the structure as each partially occupied specie occupies its
	# own position in the flattened array.
	_zs = []
	_coeffs = []
	_fcoords = []
	_dwfactors = []

	for atom in atoms:
		_zs.append(atom[0])
		try:
			c = ATOMIC_SCATTERING_PARAMS[atom_names.get(atom[0])]
		except KeyError:
			raise ValueError(
				f"Unable to calculate XRD pattern as there is no scattering coefficients for {atom[0]}."
			)
		_coeffs.append(c)
		_dwfactors.append(debye_waller_factors.get(atom_names.get(atom[0]), 0))
		_fcoords.append(atom[1:])

	zs = np.array(_zs)
	coeffs = np.array(_coeffs)
	fcoords = np.array(_fcoords)
	dwfactors = np.array(_dwfactors)
	peaks: dict[float, list[float | list[tuple[int, ...]]]] = {}
	two_thetas: list[float] = []

	for hkl, g_hkl in sorted(recip_pts, key=lambda i: (i[1], -i[0][0], -i[0][1], -i[0][2])):
		# Force miller indices to be integers.
		hkl = [int(round(i)) for i in hkl]
		if g_hkl != 0:
			# Bragg condition
			theta = asin(wavelength * g_hkl / 2)

			# s = sin(theta) / wavelength = 1 / 2d = |ghkl| / 2 (d =
			# 1/|ghkl|)
			s = g_hkl / 2

			# Store s^2 since we are using it a few times.
			s2 = s**2

			# Vectorized computation of g.r for all fractional coords and
			# hkl.
			g_dot_r = np.dot(fcoords, np.transpose([hkl])).T[0]


			# Highly vectorized computation of atomic scattering factors.
			# Equivalent non-vectorized code is::
			#
			#   for site in structure:
			#	  el = site.specie
			#	  coeff = ATOMIC_SCATTERING_PARAMS[el.symbol]
			#	  fs = el.Z - 41.78214 * s2 * sum(
			#		  [d[0] * exp(-d[1] * s2) for d in coeff])
			fs = zs - 41.78214 * s2 * np.sum(
				coeffs[:, :, 0] * np.exp(-coeffs[:, :, 1] * s2),
				axis=1,  # type: ignore
			)

			dw_correction = np.exp(-dwfactors * s2)

			# Structure factor = sum of atomic scattering factors (with
			# position factor exp(2j * pi * g.r and occupancies).
			# Vectorized computation.
			f_hkl = np.sum(fs * np.exp(2j * pi * g_dot_r) * dw_correction)

			# Lorentz polarization correction for hkl
			lorentz_factor = (1 + cos(2 * theta) ** 2) / (sin(theta) ** 2 * cos(theta))

			# Intensity for hkl is modulus square of structure factor.
			i_hkl = (f_hkl * f_hkl.conjugate()).real

			two_theta = degrees(2 * theta)

			if is_hex:
				# Use Miller-Bravais indices for hexagonal lattices.
				hkl = (hkl[0], hkl[1], -hkl[0] - hkl[1], hkl[2])
			# Deal with floating point precision issues. This combines all peaks within two_theta_tol
			two_theta_tol = 1e-5
			ind = np.where(
				np.abs(np.subtract(two_thetas, two_theta)) < two_theta_tol
			)
			if len(ind[0]) > 0:
				peaks[two_thetas[ind[0][0]]][0] += i_hkl * lorentz_factor
				peaks[two_thetas[ind[0][0]]][1].append(tuple(hkl))  # type: ignore
			else:
				d_hkl = 1 / g_hkl
				peaks[two_theta] = [i_hkl * lorentz_factor, [tuple(hkl)], d_hkl]
				two_thetas.append(two_theta)

	# Scale intensities so that the max intensity is 100.
	max_intensity = max(v[0] for v in peaks.values())
	x = []
	y = []
	hkls = []
	d_hkls = []
	for k in sorted(peaks):
		SCALED_INTENSITY_TOL = 1e-3
		if peaks[k][0] / max_intensity * 100 > SCALED_INTENSITY_TOL:  # type: ignore
			x.append(k)
			y.append(peaks[k][0])
			hkls.append(peaks[k][1])
			d_hkls.append(peaks[k][2])
	y = np.array(y)
	if scaled:
		y = y * 100 / max_intensity
	return (x, y, hkls, d_hkls)

angles = [50.0,50.0,50.0]
lengths = [4.0,4.0,4.0]
atoms = [[1,0,0,0],[1,.5,.5,.5]]
xrd_pattern = get_pattern(lengths, angles, atoms, wavelength =  1.54184, scaled = True, two_theta_range=(0, 200), hex_angle_tol = 0.01)
print(xrd_pattern[0])
print(xrd_pattern[1])

In [ ]:
plt.stem(xrd_pattern[0], xrd_pattern[1])
plt.xlim([30,75])

In [ ]:
xrd_calculator = XRDCalculator(wavelength="CuKa")
a, b, c = lengths  # lattice constants in angstroms
alpha, beta, gamma = angles  # angles in degrees
lattice = Lattice.from_parameters(a, b, c, alpha, beta, gamma)
species = ["H", "H"]  # List of species
coordinates = [[0, 0, 0], [0.5, 0.5, 0.5]]  # List of fractional coordinates
structure = Structure(lattice, species, coordinates)
pattern = xrd_calculator.get_pattern(structure)
plt.stem(pattern.x, pattern.y)
plt.xlim([30,75])